In [1]:
import pandas as pd
from dotenv import load_dotenv
import os

from langchain_community.graphs import Neo4jGraph

import warnings
warnings.filterwarnings("ignore")

load_dotenv()  # loads variables from .env


# Configuration
NEO4J_URL = os.getenv("NEO4J_URL")
NEO4J_USER = os.getenv("NEO4J_USER")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
api_key = os.getenv("OPENAI_API_KEY")
groq_api_key  = os.getenv("GROQ_API_KEY")

graph = Neo4jGraph(url=NEO4J_URL, username=NEO4J_USER, password=NEO4J_PASSWORD, database="78ce8520",
    refresh_schema=True )

In [3]:
graph.refresh_schema()

print(graph.schema)

Node properties:
Country {name: STRING, region: STRING}
YearMonth {yearMonth: INTEGER}
Relationship properties:
IMPORTED_FROM {barrels: FLOAT, value_usd: FLOAT, price_mt: FLOAT, year_month: INTEGER}
The relationships:
(:Country)-[:IMPORTS_IN]->(:YearMonth)
(:Country)-[:IMPORTED_FROM]->(:Country)


In [4]:
from langchain_core.prompts.few_shot import FewShotPromptTemplate
from langchain_core.prompts.prompt import PromptTemplate
from langchain_community.chains.graph_qa.cypher import GraphCypherQAChain
from langchain_groq import ChatGroq
from langchain_core.prompts import load_prompt

# Load prompts from YAML files
cypher_prompt = load_prompt("cypher-few-shot.yaml")
qa_prompt_template = load_prompt("qa-prompt.yaml")

llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0)

# 3. Use a powerful model. Maybe it can help in latency
chain = GraphCypherQAChain.from_llm(
    llm=llm,
    graph=graph,
    cypher_prompt=cypher_prompt,
    qa_prompt=qa_prompt_template,
    verbose=True,
    top_k=10, 
    enhanced_schema=True, 
    return_intermediate_steps=True,
    allow_dangerous_requests = True 
)

# Implementation of a "Safety Net" Wrapper
from langchain_community.callbacks.manager import get_openai_callback

def ask_sentinel(question):
    with get_openai_callback() as cb:
        try:
            response = chain.invoke({"query": question})
            
            # Print the token stats
            print(f"--- Token Usage for this Request ---")

            print(f"Prompt Tokens: {cb.prompt_tokens}")
            print(f"Completion Tokens: {cb.completion_tokens}")
            print(f"Total Tokens: {cb.total_tokens}")
            print(f"Total Cost (if applicable): ${cb.total_cost}")

            usage_pct = (cb.total_tokens / 128000) * 100
            print(f"Context Usage: {usage_pct:.2f}%")
            print(f"------------------------------------")

            if not response.get('result') or response['result'] == "I don't know the answer.":
                return "No data found for those parameters."
            
            return response['result']

        except Exception as e:
            print(f"DEBUG: Internal Error -> {str(e)}")
            return "I encountered a technical difficulty. Please try rephrasing."


In [4]:
res = ask_sentinel("Which country provided the most barrels to the USA in 202401?")
print(res)



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (importer:Country {name: 'USA'})-[:IMPORTS_IN]->(t:YearMonth {yearMonth: 202401})
MATCH (importer)-[f:IMPORTED_FROM {year_month: 202401}]->(exporter:Country)
RETURN exporter.name AS exporter,
       ROUND(sum(f.barrels) / 1000000, 2) AS total_volume_millions
ORDER BY total_volume_millions DESC
LIMIT 5
Full Context:
[{'exporter': 'Canada', 'total_volume_millions': 126.42}, {'exporter': 'Mexico', 'total_volume_millions': 16.6}, {'exporter': 'Saudi Arabia', 'total_volume_millions': 9.27}, {'exporter': 'Colombia', 'total_volume_millions': 7.71}, {'exporter': 'Brazil', 'total_volume_millions': 7.71}]

> Finished chain.
--- Token Usage for this Request ---
Prompt Tokens: 1921
Completion Tokens: 688
Total Tokens: 2609
Total Cost (if applicable): $0.0
Context Usage: 2.04%
------------------------------------
Based on total trade for Jan 2024, Canada provided the most barrels to the USA, with 126.42 million barrels.


In [5]:
res = ask_sentinel("How much barrels India imported in 202305?")
print(res)



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (importer:Country {name: 'India'})-[:IMPORTS_IN]->(tm:YearMonth {yearMonth: 202305})
MATCH (importer)-[f:IMPORTED_FROM {year_month: tm.yearMonth}]->(exporter:Country)
RETURN ROUND(sum(f.barrels)/1000000, 2) AS total_imported_barrels_millions
LIMIT 5
Full Context:
[{'total_imported_barrels_millions': 161.24}]

> Finished chain.
--- Token Usage for this Request ---
Prompt Tokens: 1831
Completion Tokens: 619
Total Tokens: 2450
Total Cost (if applicable): $0.0
Context Usage: 1.91%
------------------------------------
Based on total trade for May 2023, India imported 161.24 million barrels.


In [6]:
res = ask_sentinel("If Russians exports drop by 20%, which partner countries have the highest risk based on their number of barrels")
print(res)



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (t:YearMonth)
WITH max(t.yearMonth) / 100 AS latest_year
MATCH (importer:Country)-[:IMPORTS_IN]->(tm:YearMonth)
WHERE tm.yearMonth / 100 = latest_year
MATCH (importer)-[f:IMPORTED_FROM {year_month: tm.yearMonth}]->(exporter:Country {name: 'Russian Federation'})
WITH importer,
     sum(f.barrels) AS total_barrels
RETURN importer.name AS importer,
       ROUND(total_barrels / 1000000, 2) AS barrels_from_russia_millions,
       ROUND(total_barrels * 0.2 / 1000000, 2) AS potential_loss_if_drop_20_percent_millions
ORDER BY barrels_from_russia_millions DESC
LIMIT 5
Full Context:
[{'importer': 'China', 'barrels_from_russia_millions': 789.66, 'potential_loss_if_drop_20_percent_millions': 157.93}, {'importer': 'India', 'barrels_from_russia_millions': 669.82, 'potential_loss_if_drop_20_percent_millions': 133.96}, {'importer': 'Hungary', 'barrels_from_russia_millions': 35.07, 'potential_loss_if_drop_20_percent_millions': 7.01}, 

In [7]:
res = ask_sentinel("If Russians exports drop by 20%, which partner countries have the highest risk based on their historical reliance?")
print(res)



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (t:YearMonth)
WITH max(t.yearMonth) / 100 AS latest_year
MATCH (importer:Country)-[:IMPORTS_IN]->(tm:YearMonth)
WHERE tm.yearMonth / 100 = latest_year
MATCH (importer)-[f:IMPORTED_FROM {year_month: tm.yearMonth}]->(exporter:Country)
WITH importer,
     sum(CASE WHEN exporter.name = 'Russian Federation' THEN f.barrels ELSE 0 END) AS russian_vol,
     sum(f.barrels) AS total_vol
WHERE total_vol > 0
WITH importer,
     russian_vol,
     total_vol,
     (russian_vol / total_vol) * 100 AS reliance_percent,
     russian_vol * 0.2 AS potential_loss_vol
RETURN importer.name AS partner_country,
       ROUND(russian_vol/1000000, 2) AS russian_volume_millions,
       ROUND(total_vol/1000000, 2) AS total_import_volume_millions,
       ROUND(reliance_percent, 2) AS reliance_percent,
       ROUND(potential_loss_vol/1000000, 2) AS potential_loss_millions
ORDER BY reliance_percent DESC
LIMIT 5
Full Context:
[{'partner_country': 'Geor

In [5]:
res = ask_sentinel("Which country is most reliable on Russia for their imports?")
print(res)



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (t:YearMonth)
WITH max(t.yearMonth) / 100 AS latest_year
MATCH (importer:Country)-[:IMPORTS_IN]->(tm:YearMonth)
WHERE tm.yearMonth / 100 = latest_year
MATCH (importer)-[f:IMPORTED_FROM {year_month: tm.yearMonth}]->(exporter:Country)
WITH importer,
     sum(CASE WHEN exporter.name = 'Russian Federation' THEN f.barrels ELSE 0 END) AS russian_vol,
     sum(f.barrels) AS total_vol
WHERE total_vol > 0
RETURN importer.name AS country,
       ROUND(russian_vol / 1000000, 2) AS russian_volume_millions,
       ROUND((russian_vol / total_vol) * 100, 2) AS reliance_percent
ORDER BY reliance_percent DESC
LIMIT 5
Full Context:
[{'country': 'Georgia', 'russian_volume_millions': 0.08, 'reliance_percent': 100.0}, {'country': 'Armenia', 'russian_volume_millions': 0.0, 'reliance_percent': 100.0}, {'country': 'Azerbaijan', 'russian_volume_millions': 12.18, 'reliance_percent': 89.44}, {'country': 'Slovakia', 'russian_volume_millions': 30

In [6]:
res = ask_sentinel("Which country is most reliable on Russia for their imports for 2027?")
print(res)



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (importer:Country)-[:IMPORTS_IN]->(tm:YearMonth)
WHERE tm.yearMonth / 100 = 2027
MATCH (importer)-[f:IMPORTED_FROM {year_month: tm.yearMonth}]->(exporter:Country)
WITH importer,
     sum(CASE WHEN exporter.name = 'Russian Federation' THEN f.barrels ELSE 0 END) AS russian_vol,
     sum(f.barrels) AS total_vol
WHERE total_vol > 0
RETURN importer.name AS country,
       ROUND(russian_vol / 1000000, 2) AS russian_volume_millions,
       ROUND((russian_vol / total_vol) * 100, 2) AS reliance_percent
ORDER BY reliance_percent DESC
LIMIT 5
Full Context:
[]

> Finished chain.
--- Token Usage for this Request ---
Prompt Tokens: 1697
Completion Tokens: 976
Total Tokens: 2673
Total Cost (if applicable): $0.0
Context Usage: 2.09%
------------------------------------
Based on total trade for 2027, there is no data available to determine which country is most reliable on Russia for their imports.


In [ ]:
res = ask_sentinel("Find instances where an importer is importing the same commodity from two different exporter at a price difference of >30%.")
print(res)

In [7]:
res = ask_sentinel("Who is the largest exporter of oil?")
print(res)



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (t:YearMonth)
WITH max(t.yearMonth) / 100 AS latest_year
MATCH (importer:Country)-[:IMPORTS_IN]->(tm:YearMonth)
WHERE tm.yearMonth / 100 = latest_year
MATCH (importer)-[f:IMPORTED_FROM {year_month: tm.yearMonth}]->(exporter:Country)
RETURN exporter.name AS exporter,
       ROUND(sum(f.barrels)/1000000, 2) AS total_volume_millions
ORDER BY total_volume_millions DESC
LIMIT 5
Full Context:
[{'exporter': 'Saudi Arabia', 'total_volume_millions': 2145.19}, {'exporter': 'Russian Federation', 'total_volume_millions': 1571.25}, {'exporter': 'Canada', 'total_volume_millions': 1518.28}, {'exporter': 'USA', 'total_volume_millions': 1321.01}, {'exporter': 'United Arab Emirates', 'total_volume_millions': 1318.54}]

> Finished chain.
--- Token Usage for this Request ---
Prompt Tokens: 1787
Completion Tokens: 824
Total Tokens: 2611
Total Cost (if applicable): $0.0
Context Usage: 2.04%
------------------------------------
Based on tot

In [8]:
res = ask_sentinel("If there is a war in the middle east then which countries imports will be affected and by how much?\
Will India be affected?")
print(res)



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (t:YearMonth)
WITH max(t.yearMonth) / 100 AS latest_year
MATCH (imp:Country)-[:IMPORTS_IN]->(tm:YearMonth)
WHERE tm.yearMonth / 100 = latest_year
MATCH (imp)-[f:IMPORTED_FROM {year_month: tm.yearMonth}]->(exp:Country {region: 'Middle East'})
WITH imp.name AS importer, ROUND(sum(f.barrels) / 1000000, 2) AS volume
ORDER BY volume DESC
WITH collect({importer: importer, volume: volume})[0..5] AS top5,
     any(x IN collect({importer: importer, volume: volume}) WHERE x.importer = 'India') AS india_affected
UNWIND top5 AS row
RETURN row.importer AS importer, row.volume AS affected_volume_millions, india_affected
Full Context:
[{'importer': 'China', 'affected_volume_millions': 1789.72, 'india_affected': True}, {'importer': 'Japan', 'affected_volume_millions': 817.61, 'india_affected': True}, {'importer': 'India', 'affected_volume_millions': 809.27, 'india_affected': True}, {'importer': 'Rep. of Korea', 'affected_volume_milli

In [12]:
res = ask_sentinel("Largest importer of crude oil in South East Asia?")
print(res)



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (t:YearMonth)
WITH max(t.yearMonth) / 100 AS latest_year
MATCH (importer:Country)-[:IMPORTS_IN]->(tm:YearMonth)
WHERE tm.yearMonth / 100 = latest_year AND importer.region = 'South East Asia'
MATCH (importer)-[f:IMPORTED_FROM {year_month: tm.yearMonth}]->(exporter:Country)
RETURN importer.name AS importer_country,
       ROUND(sum(f.barrels) / 1000000, 2) AS total_import_volume_millions
ORDER BY total_import_volume_millions DESC
LIMIT 5
Full Context:
[{'importer_country': 'Thailand', 'total_import_volume_millions': 402.75}, {'importer_country': 'Singapore', 'total_import_volume_millions': 321.91}, {'importer_country': 'Malaysia', 'total_import_volume_millions': 162.69}, {'importer_country': 'Indonesia', 'total_import_volume_millions': 124.99}, {'importer_country': 'Philippines', 'total_import_volume_millions': 47.6}]

> Finished chain.
--- Token Usage for this Request ---
Prompt Tokens: 1922
Completion Tokens: 800
Tota

In [13]:
res = ask_sentinel("Largest exporter of crude oil in South East Asia?")
print(res)



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (t:YearMonth)
WITH max(t.yearMonth) / 100 AS latest_year
MATCH (importer:Country)-[:IMPORTS_IN]->(tm:YearMonth)
WHERE tm.yearMonth / 100 = latest_year
MATCH (importer)-[f:IMPORTED_FROM {year_month: tm.yearMonth}]->(exporter:Country {region: 'South East Asia'})
RETURN exporter.name AS exporter,
       ROUND(sum(f.barrels) / 1000000, 2) AS total_exported_millions
ORDER BY total_exported_millions DESC
LIMIT 5
Full Context:
[{'exporter': 'Malaysia', 'total_exported_millions': 595.5}, {'exporter': 'Indonesia', 'total_exported_millions': 31.07}, {'exporter': 'Brunei Darussalam', 'total_exported_millions': 24.73}, {'exporter': 'Viet Nam', 'total_exported_millions': 17.47}, {'exporter': 'Thailand', 'total_exported_millions': 6.5}]

> Finished chain.
--- Token Usage for this Request ---
Prompt Tokens: 1922
Completion Tokens: 859
Total Tokens: 2781
Total Cost (if applicable): $0.0
Context Usage: 2.17%
--------------------------

In [9]:
res = ask_sentinel("Hoe to bake French bread?")
print(res)



> Entering new GraphCypherQAChain chain...
Generated Cypher:
I’m sorry, but I can only help with converting questions about trade data into Cypher queries.
DEBUG: Internal Error -> {neo4j_code: Neo.ClientError.Statement.SyntaxError} {message: Invalid input 'I': expected 'ALTER', 'ORDER BY', 'CALL', 'CREATE', 'LOAD CSV', 'START DATABASE', 'STOP DATABASE', 'DEALLOCATE', 'DELETE', 'DENY', 'DETACH', 'DROP', 'DRYRUN', 'FILTER', 'FINISH', 'FOREACH', 'GRANT', 'INSERT', 'LET', 'LIMIT', 'MATCH', 'MERGE', 'NODETACH', 'OFFSET', 'OPTIONAL', 'REALLOCATE', 'REMOVE', 'RENAME', 'RETURN', 'REVOKE', 'ENABLE SERVER', 'SET', 'SHOW', 'SKIP', 'TERMINATE', 'UNWIND', 'USE', 'WHEN', 'WITH' or '{' (line 1, column 1 (offset: 0))
"I’m sorry, but I can only help with converting questions about trade data into Cypher queries."
 ^} {gql_status: 42001} {gql_status_description: error: syntax error or access rule violation - invalid syntax}
I encountered a technical difficulty. Please try rephrasing.


In [10]:
res = ask_sentinel("If there is a conflict between Iran and Israel then which countries imports will be affected?")
print(res)

#The LLM does not inherently know that a conflict between Iran and Israel might lead to a blockade of the Strait of Hormuz,
#  which would affect all Middle Eastern oil (Saudi, UAE, Kuwait, etc.). 
#It only knows what is in the relationship lines.


# 10. GEOPOLITICAL IMPACT: If a question mentions a conflict or 'risk' in the Middle East (e.g., Iran, Israel, Red Sea):
        # a) First, query direct trade for those specific countries.
        # b) Second, automatically query the total imports from the broader region: 
        #    ['Saudi Arabia', 'United Arab Emirates', 'Iraq', 'Kuwait', 'Qatar', 'Oman'].
        # c) Return BOTH sets of data so the user can see the direct and regional exposure.



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (t:YearMonth)
WITH max(t.yearMonth) / 100 AS latest_year
MATCH (importer:Country)-[:IMPORTS_IN]->(tm:YearMonth)
WHERE tm.yearMonth / 100 = latest_year
MATCH (importer)-[f:IMPORTED_FROM {year_month: tm.yearMonth}]->(exporter:Country)
WHERE exporter.name IN ['Iran','Israel']
RETURN importer.name AS affected_importer,
       ROUND(sum(f.barrels)/1000000, 2) AS total_volume_millions,
       ROUND(avg(f.price_mt), 2) AS average_price
ORDER BY total_volume_millions DESC
LIMIT 5
Full Context:
[{'affected_importer': 'Netherlands', 'total_volume_millions': 3.94, 'average_price': 615.51}]

> Finished chain.
--- Token Usage for this Request ---
Prompt Tokens: 1731
Completion Tokens: 1162
Total Tokens: 2893
Total Cost (if applicable): $0.0
Context Usage: 2.26%
------------------------------------
Based on total trade for 2023, the Netherlands imports will be affected, with a total volume of 3.94 million barrels.
